# Avaliação Quantitativa da PoC — SSCAD 2026

Este relatório documenta os dois experimentos de avaliação quantitativa da PoC de
Data Lakehouse (medalhão Bronze/Silver/Gold sobre Iceberg, orquestrado por Airflow,
processado por Spark, consultado via Trino). O objetivo é produzir evidência
empírica mínima e defensável para a Trilha Principal do SSCAD — a Seção 5 do artigo
original não tinha números, só a arquitetura; este notebook preenche essa lacuna.

**Por que dois experimentos, e por que esses dois:**

- **E1 — Escalabilidade com o volume**: Mede como a duração de cada estágio (Bronze, Silver, Gold) cresce conforme o número de registros ingeridos aumenta em duas ordens de grandeza (8×10³ → 8×10⁵ registros).
- **E2 — Speedup com paralelismo**: Mede quanto a DAG mais pesada da pipeline (`dag_silver_transform`, 8 transformações) acelera quando o Spark recebe mais núcleos.

Os dois foram desenhados para reportar grandezas **relativas** (formato da curva em E1, speedup adimensional em E2) — isso é deliberado: como o hardware é um único computador pessoal (não um cluster), qualquer número absoluto seria pouco generalizável.

Dados brutos gerados por `scripts/orquestrar_protocolo.sh` (que chama `scripts/experimento.sh` e `scripts/coleta_metricas_trino.sh` por repetição) e lidos aqui de `resultados/*.csv`. Ver `resultados/log_execucao.txt` para o log completo da execução e `resultados/ambiente.txt` para as specs de hardware.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULT_DIR = Path("resultados")
FIG_DIR = Path("figuras")
FIG_DIR.mkdir(exist_ok=True)


COR_BRONZE = "#CD7F32"   
COR_SILVER = "#C0C0C0"   
COR_GOLD   = "#D4AF37"   
CORES_ESTAGIO = {"dag_ingestao_bronze": COR_BRONZE, "dag_silver_transform": COR_SILVER, "dag_gold_refresh": COR_GOLD}
MARCADORES_ESTAGIO = {"dag_ingestao_bronze": "o", "dag_silver_transform": "s", "dag_gold_refresh": "^"}
LINHAS_ESTAGIO = {"dag_ingestao_bronze": "-", "dag_silver_transform": "--", "dag_gold_refresh": ":"}
NOMES_ESTAGIO = {"dag_ingestao_bronze": "Bronze", "dag_silver_transform": "Silver", "dag_gold_refresh": "Gold"}

# Rampa sequencial (ordinal) de azul p/ E2 — C1 < C2 < C4 cores
RAMPA_CORES_E2 = {1: "#86b6ef", 2: "#2a78d6", 4: "#0d366b"}

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 11,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


# --- Intervalo de confianca (t de Student) ---------------------------------
# scipy nao esta no venv, entao a tabela de t bicaudal 95% (df -> t) fica
# embutida. Reportamos IC 95% AO LADO do desvio-padrao: com poucas repeticoes,
# o std cru subestima a incerteza da media; o IC = t(0.975, n-1) * std/sqrt(n)
# a corrige pelo tamanho da amostra. df ausente na tabela (>30) cai no z=1.96.
_T95 = {
    1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365,
    8: 2.306, 9: 2.262, 10: 2.228, 11: 2.201, 12: 2.179, 13: 2.160, 14: 2.145,
    15: 2.131, 16: 2.120, 17: 2.110, 18: 2.101, 19: 2.093, 20: 2.086,
    21: 2.080, 22: 2.074, 23: 2.069, 24: 2.064, 25: 2.060, 26: 2.056,
    27: 2.052, 28: 2.048, 29: 2.045, 30: 2.042,
}

def t95(n):
    """Multiplicador t bicaudal 95% para uma amostra de tamanho n (df = n-1)."""
    df = int(n) - 1
    if df < 1:
        return float("nan")
    return _T95.get(df, 1.96)

def add_ic95(agg, col_std="std", col_n="count", col_mean="mean"):
    """Acrescenta erro-padrao (sem) e meia-largura do IC 95% (ic95) a um DataFrame
    ja agregado com colunas de media, desvio-padrao e n. O IC completo da media
    e [mean - ic95, mean + ic95]."""
    agg = agg.copy()
    agg["sem"] = agg[col_std] / np.sqrt(agg[col_n])
    agg["ic95"] = agg[col_n].map(t95) * agg["sem"]
    return agg


## 1. Ambiente experimental

In [2]:
ambiente_path = RESULT_DIR / "ambiente.txt"
if ambiente_path.exists():
    print(ambiente_path.read_text())
else:
    print("resultados/ambiente.txt ainda nao existe — rode scripts/orquestrar_protocolo.sh primeiro.")

data_inicio_utc: 2026-07-21T13:18:41Z

--- CPU ---
CPU(s):                                    20
Thread(s) per núcleo:                      2

--- Memoria ---
               total       usada       livre    compart.  buff/cache  disponível
Mem.:           31Gi        13Gi       6,8Gi       1,0Gi        11Gi        17Gi
Swap:          2,0Gi          0B       2,0Gi

--- Disco (repo) ---
Sist. Arq.      Tam. Usado Disp. Uso% Montado em
/dev/nvme0n1p2  468G  336G  109G  76% /

--- Docker ---
Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
Docker Compose version v5.3.1

--- SO ---
Linux saiti-HP-Elite-SFF-600-G9-Desktop-PC 6.8.0-134-generic #134-Ubuntu SMP PREEMPT_DYNAMIC Fri Jun 26 18:43:11 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

--- Imagens da stack ---
CONTAINER               REPOSITORY                     TAG                 PLATFORM            IMAGE ID            SIZE                CREATED
dlh_airflow_init        artigo_dlh-airflow-init        latest              linux/amd64    

**Controles de validade aplicados** (checklist do protocolo) — por que existem:
cada um remove uma fonte específica de ruído que poderia contaminar a medição.

- **Reset total** (`docker compose down -v` + `up -d`) antes de cada repetição —
  sem isso, a Bronze acumularia registros de repetições anteriores e o volume
  "V1" da terceira repetição não seria mais o mesmo volume "V1" da primeira.
- **Governança desligada** (OpenMetadata/Elasticsearch) durante E1/E2 — esses
  serviços consomem CPU/RAM continuamente e não participam da medição; deixá-los
  ligados enviesaria a duração para cima sem relação com o que está sendo testado.
- `dag_iceberg_maintenance` e `dag_trino_governance` permanecem pausadas durante
  todo o protocolo pelo mesmo motivo.
- **Pausa/despausa em torno de cada disparo**: neste ambiente (Airflow 2.9 +
  LocalExecutor) uma DAG pausada bloqueia até runs disparadas manualmente, então
  é preciso despausar para medir — mas isso também reativa o agendamento cron
  próprio da DAG (`*/5`, `*/15`, `*/20` min), que poderia disparar uma segunda
  execução concorrente disputando o mesmo Spark. A mitigação: despausar,
  disparar, esperar terminar **por completo**, só então pausar de novo (pausar
  no meio trava as tasks restantes nesta versão do Airflow — foi preciso
  descobrir isso na prática). A proteção contra o cron da própria DAG concorrer
  vem do `max_active_runs=1` de cada DAG.
- **Seeds fixas e distintas por repetição** (`--seed`, ver `scripts/gerar_dados.py`)
  — reprodutibilidade: qualquer pessoa rodando o mesmo `--seed` gera exatamente
  os mesmos dados sintéticos.

## 2. Dados coletados

In [3]:
def carregar_csv(nome, **kw):
    p = RESULT_DIR / nome
    if not p.exists():
        print(f"aviso: {p} nao encontrado")
        return pd.DataFrame()
    return pd.read_csv(p, **kw)

duracoes = carregar_csv("duracoes.csv")
tasks = carregar_csv("task_durations.csv")
contagens = carregar_csv("contagens.csv")
tamanhos = carregar_csv("tamanhos_arquivos.csv")
snapshots = carregar_csv("snapshots.csv")

print(f"duracoes: {len(duracoes)} linhas | tasks: {len(tasks)} | contagens: {len(contagens)} | tamanhos: {len(tamanhos)} | snapshots: {len(snapshots)}")

falhas = duracoes[duracoes["segundos"] == "FALHOU"] if not duracoes.empty else pd.DataFrame()
if len(falhas):
    print(f"\n{len(falhas)} repeticoes FALHARAM (nao entram nas medias abaixo):")
    display(falhas)
else:
    print("\nNenhuma repeticao falhou.")

if not duracoes.empty:
    duracoes = duracoes[duracoes["segundos"] != "FALHOU"].copy()
    duracoes["segundos"] = duracoes["segundos"].astype(float)

duracoes: 54 linhas | tasks: 252 | contagens: 162 | tamanhos: 144 | snapshots: 144

Nenhuma repeticao falhou.


## 3. E1 — Escalabilidade com o volume

**Objetivo deste teste:** verificar se a arquitetura medalhão (Bronze → Silver → Gold) degrada de forma abrupta/imprevisível conforme o volume de dados cresce, ou se o crescimento é suave e explicável.

**O que foi medido:** a duração de cada uma das 3 DAGs de produção (`dag_ingestao_bronze`, `dag_silver_transform`, `dag_gold_refresh`), disparadas em sequência, para 3 volumes de dados (V1=8 mil, V2=80 mil, V3=800 mil registros — 2/20/200 lotes × 500 registros × 8 tipos de dado), com 3 repetições por volume e reset total do ambiente entre cada uma.

**Hipótese/expectativa declarada no protocolo:** crescimento aproximadamente linear com o volume, mas só depois de um "piso" de overhead fixo (custo de sessão Spark + orquestração Airflow) — a hipótese contrária seria um estágio que não escala nada (aparente sinal de gargalo estrutural) ou que escala pior que linear (aparente sinal de problema algorítmico, ex. um shuffle mal particionado).

In [ ]:
e1 = duracoes[duracoes["tag"].str.match(r"^V\d+_rep\d+$")].copy() if not duracoes.empty else pd.DataFrame()

if len(e1):
    e1_agg = (
        e1.groupby(["dag", "volume_total"])["segundos"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("volume_total")
    )
    # Erro-padrao e IC 95% (t de Student) ao lado do std cru.
    e1_agg = add_ic95(e1_agg)
    display(e1_agg)
else:
    print("Sem dados de E1 ainda.")


In [ ]:
if len(e1):
    fig, ax = plt.subplots(figsize=(7, 5))
    for dag in ["dag_ingestao_bronze", "dag_silver_transform", "dag_gold_refresh"]:
        d = e1_agg[e1_agg["dag"] == dag]
        if not len(d):
            continue
        # Barras de erro = IC 95% (t de Student); o std cru esta na tabela acima.
        ax.errorbar(
            d["volume_total"], d["mean"], yerr=d["ic95"].fillna(0),
            marker=MARCADORES_ESTAGIO[dag], linestyle=LINHAS_ESTAGIO[dag],
            color=CORES_ESTAGIO[dag], label=NOMES_ESTAGIO[dag],
            capsize=4, linewidth=1.8, markersize=7,
        )
    # x em log (volume varre decadas); y linear para NAO achatar visualmente a
    # diferenca entre estagios quase planos (Gold ~122s) e o crescimento do Bronze.
    ax.set_xscale("log")
    ax.set_xlabel("Volume total (registros)")
    ax.set_ylabel("Duração da DAG (s)")
    ax.set_title("E1 — Duração por estágio vs. volume de dados")
    ax.legend(frameon=False, title="barras = IC 95%")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_e1_escalabilidade.png", dpi=200)
    plt.show()

    # Throughput derivado (Secao 2.4 do protocolo)
    e1_agg["throughput_reg_s"] = e1_agg["volume_total"] / e1_agg["mean"]
    print("\nThroughput por estagio (registros/s):")
    display(e1_agg[["dag", "volume_total", "mean", "throughput_reg_s"]])
    print(
        "Ressalva: para Silver/Gold (duracao quase constante, dominada por overhead fixo), "
        "throughput crescente com o volume reflete amortizacao desse custo fixo, nao ganho "
        "de vazao real de processamento — so no Bronze o throughput mede taxa de fato."
    )
else:
    print("Sem dados de E1 ainda — rode o protocolo primeiro.")


In [ ]:
if len(e1):
    linhas = []
    for dag in ["dag_ingestao_bronze", "dag_silver_transform", "dag_gold_refresh"]:
        d = e1_agg[e1_agg["dag"] == dag].sort_values("volume_total")
        if len(d) < 2:
            continue
        t_min, t_max = d["mean"].iloc[0], d["mean"].iloc[-1]
        v_min, v_max = d["volume_total"].iloc[0], d["volume_total"].iloc[-1]
        cresc_tempo = t_max / t_min
        cresc_volume = v_max / v_min
        linhas.append((NOMES_ESTAGIO[dag], v_min, v_max, t_min, t_max, cresc_volume, cresc_tempo))

    n_rep = int(e1_agg["count"].max())

    print("Comentário sobre os resultados de E1")
    print("=" * 60)
    for nome, v0, v1, t0, t1, cv, ct in linhas:
        print(
            f"- {nome}: volume cresceu {cv:.0f}x ({v0:,} -> {v1:,} registros) "
            f"e a duração cresceu {ct:.2f}x ({t0:.0f}s -> {t1:.0f}s)."
        )
    print()
    mais_sensivel = max(linhas, key=lambda x: x[6])
    menos_sensivel = min(linhas, key=lambda x: x[6])
    print(
        f"O estágio mais sensível ao volume foi {mais_sensivel[0]} "
        f"({mais_sensivel[6]:.2f}x de crescimento na duração para {mais_sensivel[5]:.0f}x "
        f"de crescimento no volume). O menos sensível foi {menos_sensivel[0]} "
        f"({menos_sensivel[6]:.2f}x), praticamente plano."
    )
    print(
        "\nLeitura: Silver e Gold ficam dominados pelo custo fixo de inicializar uma "
        "sessão Spark por task (8 tasks sequenciais na Silver, 4 na Gold) — esse custo "
        "não depende de quantos registros existem, só de quantas vezes uma JVM Spark "
        "precisa subir. Só o Bronze cresce de forma visível, provavelmente porque o "
        "número de arquivos JSON lidos do MinIO escala 1:1 com o volume (mais objetos "
        "para listar/ler via S3A), diferente de Silver/Gold onde o gargalo é "
        "orquestração, não volume de dados processado. Isso é o 'piso de overhead "
        "fixo' que o protocolo antecipava — é um achado honesto sobre o custo de "
        "coordenação da arquitetura em cargas pequenas, não um defeito da PoC."
    )
    print(
        "\nRessalva importante sobre o alcance de E1: como a duração cresce muito menos "
        "que o volume em TODOS os estágios (o mais sensível cresce só "
        f"{mais_sensivel[6]:.1f}x para {mais_sensivel[5]:.0f}x de volume), nesses volumes "
        "a pipeline nunca sai do regime dominado por overhead fixo — nem no maior ponto o "
        "processamento de dados passa a dominar. Ou seja, E1 é evidência de que a "
        "arquitetura é ROBUSTA e PREVISÍVEL nesse regime (não quebra, não degrada "
        "abruptamente), mas não mede a escalabilidade do processamento em si, que só "
        "apareceria em volumes onde o custo de dados supere o de coordenação (ver V4, "
        "Seção 7)."
    )
    print(
        f"\nNota estatística: cada ponto é a média de {n_rep} repetições, com a duração "
        "agora cronometrada em milissegundos (date +%s.%N) — sem a granularidade de "
        "segundos inteiros da versão anterior, que produzia std≈0 artificial nos "
        "estágios quase-planos. A tabela reporta desvio-padrão (std) e IC 95% (t de "
        "Student) lado a lado: o std descreve a dispersão observada entre repetições; o "
        "IC 95% (barras da Figura E1) quantifica a incerteza da média corrigida pelo "
        f"tamanho da amostra (n={n_rep}). Std muito baixo aqui reflete um ambiente "
        "determinístico (mesmo hardware, reset total entre reps), não mais um artefato "
        "de resolução do timer."
    )
else:
    print("Sem dados de E1 ainda.")


## 4. E2 — Speedup com paralelismo (Silver, volume fixo V3)

**Objetivo deste teste:** a Silver é a DAG mais pesada da pipeline (8 transformações
Bronze→Silver, uma por tipo de dado). E2 pergunta se ela usa mais recursos quando
mais recursos são oferecidos — um teste direto de que a escolha de Spark como
motor de processamento se justifica (ou não) nesse volume de trabalho.

**O que foi medido:** a duração da `dag_silver_transform` com volume fixo em V3
(800 mil registros, o maior de E1), variando `spark.cores.max`/`spark.executor.cores`
em 1 (baseline), 2 e 4 núcleos — 3 repetições por configuração. Esse é o desenho
**E2a (intra-job)**: cada um dos 8 `spark-submit` recebe mais núcleos, sem mudar a
estrutura da DAG (as 8 tasks continuam encadeadas sequencialmente).

**Hipótese/expectativa declarada no protocolo:** speedup sublinear é esperado (a
carga é pequena para HPC), mas o valor exato — e principalmente o motivo do
resultado — é o que importa reportar, inclusive se a eficiência cair bastante.

In [ ]:
e2 = duracoes[duracoes["tag"].str.match(r"^C\d+_rep\d+$") & (duracoes["dag"] == "dag_silver_transform")].copy() if not duracoes.empty else pd.DataFrame()

if len(e2):
    e2["cores"] = e2["tag"].str.extract(r"^C(\d+)_rep\d+$").astype(int)
    e2_agg = e2.groupby("cores")["segundos"].agg(["mean", "std", "count"]).reset_index().sort_values("cores")
    # Erro-padrao e IC 95% (t de Student) da duracao, ao lado do std cru.
    e2_agg = add_ic95(e2_agg)

    t_baseline = e2_agg.loc[e2_agg["cores"] == 1, "mean"]
    sem_baseline = e2_agg.loc[e2_agg["cores"] == 1, "sem"]
    if len(t_baseline):
        t_baseline = t_baseline.iloc[0]
        sem_baseline = sem_baseline.iloc[0]
        e2_agg["speedup"] = t_baseline / e2_agg["mean"]
        e2_agg["eficiencia"] = e2_agg["speedup"] / e2_agg["cores"]
        # IC 95% do speedup (razao T(C1)/T(CN)): propagacao de erro relativo,
        # (sem/mean) somado em quadratura entre baseline e ponto atual.
        rel = np.sqrt((sem_baseline / t_baseline) ** 2 + (e2_agg["sem"] / e2_agg["mean"]) ** 2)
        e2_agg["speedup_ic95"] = e2_agg["count"].map(t95) * e2_agg["speedup"] * rel
    display(e2_agg)
else:
    print("Sem dados de E2 ainda.")


In [ ]:
if len(e2):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    ax = axes[0]
    cores_ordenados = sorted(e2_agg["cores"].unique())
    # Barras de erro = IC 95% (t de Student); std cru esta na tabela acima.
    ax.bar(
        [str(c) for c in cores_ordenados],
        [e2_agg.set_index("cores").loc[c, "mean"] for c in cores_ordenados],
        yerr=[e2_agg.set_index("cores").loc[c, "ic95"] for c in cores_ordenados],
        color=[RAMPA_CORES_E2.get(c, COR_SILVER) for c in cores_ordenados],
        capsize=5,
    )
    ax.set_xlabel("Cores do worker Spark (spark.cores.max)")
    ax.set_ylabel("Duração da dag_silver_transform (s)")
    ax.set_title("Duração por nº de cores (barras = IC 95%)")

    ax2 = axes[1]
    if "speedup" in e2_agg.columns:
        ax2.errorbar(
            e2_agg["cores"], e2_agg["speedup"],
            yerr=e2_agg["speedup_ic95"].fillna(0),
            marker="o", color=COR_SILVER, capsize=4, label="Speedup S(N)",
        )
        ax2.plot(e2_agg["cores"], e2_agg["cores"], linestyle="--", color="#898781", label="Speedup ideal (linear)")
        ax2.set_xlabel("Cores (N)")
        ax2.set_ylabel("Speedup S(N) = T(C1) / T(CN)")
        ax2.set_title("Speedup vs. nº de cores (barras = IC 95%)")
        ax2.legend(frameon=False)
        ax2.set_xticks(sorted(e2_agg["cores"].unique()))

    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_e2_speedup.png", dpi=200)
    plt.show()
else:
    print("Sem dados de E2 ainda — rode o protocolo primeiro.")


In [ ]:
tarefas_agg = None
if len(tasks):
    tarefas_e2 = tasks[tasks["tag"].str.match(r"^C\d+_rep\d+$") & (tasks["dag"] == "dag_silver_transform")].copy()
    if len(tarefas_e2):
        tarefas_e2["cores"] = tarefas_e2["tag"].str.extract(r"^C(\d+)_rep\d+$").astype(int)
        tarefas_agg = tarefas_e2.groupby(["cores", "task_id"])["duration_s"].mean().unstack("task_id")
        display(tarefas_agg.round(1))
    else:
        print("Sem duracoes por task para E2 ainda.")

task_id,silver_gps,silver_material,silver_obstaculo,silver_paf,silver_pessoal,silver_relt_intel,silver_seg_area,silver_sensor
cores,,,,,,,,
1,15.5,12.3,13.1,13.0,13.4,13.0,13.2,12.5
2,13.7,11.5,12.0,12.0,12.2,12.0,12.0,11.5
4,12.7,12.4,11.6,11.6,11.4,11.6,11.7,11.1


In [ ]:
if len(e2) and "speedup" in e2_agg.columns:
    linha_max = e2_agg.loc[e2_agg["cores"] == e2_agg["cores"].max()]
    n_max = int(linha_max["cores"].iloc[0])
    speedup_max = linha_max["speedup"].iloc[0]
    efic_max = linha_max["eficiencia"].iloc[0]
    ic_max = linha_max["speedup_ic95"].iloc[0]
    n_rep = int(e2_agg["count"].max())

    print("Comentário sobre os resultados de E2")
    print("=" * 60)
    for _, row in e2_agg.iterrows():
        base = f"- {int(row['cores'])} core(s): {row['mean']:.1f}s (±{row['ic95']:.1f} IC95)"
        if "speedup" in row:
            base += (
                f" | speedup {row['speedup']:.2f}x (±{row['speedup_ic95']:.2f}) "
                f"| eficiência {row['eficiencia']:.0%}"
            )
        print(base)
    print()
    print(
        f"Com {n_max} núcleos o speedup medido foi de apenas {speedup_max:.2f}x "
        f"(IC 95% ±{ic_max:.2f}; eficiência {efic_max:.0%} do ideal linear). "
    )
    # Um speedup so e estatisticamente distinguivel de 'sem ganho' se o IC 95%
    # nao inclui 1.0 — leitura honesta que n=3 nao permitia fazer.
    if speedup_max - ic_max > 1.0:
        print(
            f"O IC 95% do speedup a {n_max} núcleos ([{speedup_max - ic_max:.2f}, "
            f"{speedup_max + ic_max:.2f}]) não inclui 1.0, então há ganho real, ainda que "
            f"pequeno — distinção que só {n_rep} repetições permitem afirmar."
        )
    else:
        print(
            f"O IC 95% do speedup a {n_max} núcleos ([{speedup_max - ic_max:.2f}, "
            f"{speedup_max + ic_max:.2f}]) ainda inclui 1.0, ou seja, mesmo com "
            f"{n_rep} repetições o ganho não é estatisticamente distinguível de 'sem "
            f"aceleração' — coerente com um gargalo dominado por overhead, não por "
            f"processamento paralelizável."
        )
    if tarefas_agg is not None:
        media_por_task = tarefas_agg.mean(axis=1)
        print(
            f"Olhando o detalhamento por task: a duração média de cada uma das 8 tasks "
            f"individuais varia pouco entre 1 e {n_max} núcleos "
            f"({media_por_task.min():.1f}s a {media_por_task.max():.1f}s), porque cada "
            f"`spark-submit` paga um custo fixo de subida de JVM/sessão Spark que não "
            f"encolhe com mais núcleos — esse custo domina o tempo de cada task nesse "
            f"volume de dados, não o processamento em si."
        )
    print(
        "\nLeitura: o speedup baixo não é sinal de que Spark foi má escolha, é sinal de "
        "que o gargalo desta DAG é orquestração (8 spark-submit sequenciais, cada um "
        "pagando startup fixo), não paralelismo de dados dentro de cada job — "
        "'spark.cores.max' só acelera a fração de tempo gasta processando dados, que é "
        "pequena perto do overhead de sessão neste volume. Isso aponta para E2b "
        "(paralelizar as 8 tasks entre si, não só dar mais núcleos a cada uma) como o "
        "próximo experimento com potencial de ganho real — ver Seção 7."
    )
else:
    print("Sem dados de E2 ainda.")


## 5. Tabela descritiva — contagens, deduplicação, tamanho e snapshots

**Objetivo desta tabela:** a Seção 5 do artigo original descrevia a arquitetura mas
não trazia nenhum número sobre o que ela de fato produz — quantos registros
sobrevivem a cada camada, se a deduplicação por chave natural funciona, quanto
espaço as tabelas ocupam e se o versionamento Iceberg (snapshots, time travel) está
realmente acontecendo. Esta tabela fecha essa lacuna.

In [11]:
if len(contagens):
    tag_ref = sorted([t for t in contagens["tag"].unique() if t.startswith("V3_")])
    tag_ref = tag_ref[-1] if tag_ref else contagens["tag"].iloc[-1]
    print(f"Usando tag de referencia: {tag_ref}")

    c = contagens[contagens["tag"] == tag_ref].copy()
    bronze_total = c[c["zona"] == "bronze"]["registros"].sum()
    silver = c[c["zona"] == "silver"].copy()
    silver = silver.rename(columns={"tabela": "tipo", "registros": "registros_silver"})

    tam = tamanhos[tamanhos["tag"] == tag_ref].copy()
    tam["tipo"] = tam["tabela"].str.replace("silver.", "", regex=False)

    snap = snapshots[snapshots["tag"] == tag_ref].copy()
    snap["tipo"] = snap["tabela"].str.replace("silver.", "", regex=False)

    # A Bronze e uma unica tabela mista ('dados') com os 8 tipos repartidos igualmente
    # pelo gerador (mesmo n de registros por tipo). O denominador honesto da dedup de
    # CADA tipo e a fatia da Bronze daquele tipo (total / n_tipos), NAO o total da Bronze:
    # usar o total inflaria toda taxa em ~1-1/n_tipos so pelo efeito da mistura.
    n_tipos = silver["tipo"].nunique()
    registros_bronze_tipo = bronze_total / n_tipos

    desc = silver.merge(tam[["tipo", "n_arquivos", "tamanho_mb"]], on="tipo", how="left")
    desc = desc.merge(snap[["tipo", "n_snapshots"]], on="tipo", how="left")
    desc["registros_bronze_tipo"] = registros_bronze_tipo
    desc["taxa_dedup_pct"] = (1 - desc["registros_silver"] / desc["registros_bronze_tipo"]) * 100
    desc = desc[["tipo", "registros_bronze_tipo", "registros_silver", "taxa_dedup_pct", "n_arquivos", "tamanho_mb", "n_snapshots"]]
    display(desc.round(2))
    print(
        f"\n(Bronze total = {int(bronze_total):,} registros, repartidos entre {n_tipos} tipos "
        f"=> {int(registros_bronze_tipo):,} por tipo, usado como denominador da dedup.)"
    )
    desc.to_csv(RESULT_DIR / f"tabela_descritiva_{tag_ref}.csv", index=False)
else:
    desc = pd.DataFrame()
    print("Sem dados de contagens ainda.")

Usando tag de referencia: V3_rep3


,tipo,registros_bronze_tipo,registros_silver,taxa_dedup_pct,n_arquivos,tamanho_mb,n_snapshots
0,gps,100000.0,100000,0.0,7,2.54,2
1,sensor,100000.0,105,99.9,7,0.04,2
2,relt_intel,100000.0,100000,0.0,7,6.38,2
3,paf,100000.0,100000,0.0,7,4.25,2
4,obstaculo,100000.0,100000,0.0,7,4.42,2
5,seg_area,100000.0,100000,0.0,7,4.28,2
6,pessoal,100000.0,100000,0.0,7,3.05,2
7,material,100000.0,100000,0.0,7,0.95,2



(Bronze total = 800,000 registros, repartidos entre 8 tipos => 100,000 por tipo, usado como denominador da dedup.)


In [12]:
# Comentario sobre a tabela descritiva.
if len(desc):
    print("Comentário sobre a tabela descritiva")
    print("=" * 60)
    print(
        "Leitura da coluna 'taxa_dedup_pct': o denominador agora é a fatia da Bronze de "
        "cada tipo (total / n_tipos), então a taxa reflete deduplicação real por tipo, "
        "não o artefato de misturar 8 tipos numa Bronze só. Taxa ~0% = o tipo usa "
        "identificador gerado por execução (sempre único), então o MERGE INTO da Silver "
        "não tem o que colapsar; taxa alta = deduplicação real por chave natural."
    )
    linha_max = desc.loc[desc["taxa_dedup_pct"].idxmax()]
    linha_min = desc.loc[desc["taxa_dedup_pct"].idxmin()]
    if linha_max["taxa_dedup_pct"] - linha_min["taxa_dedup_pct"] > 1:
        print(
            f"\n'{linha_max['tipo']}' se destaca com {linha_max['taxa_dedup_pct']:.1f}% de "
            f"dedup — {int(linha_max['registros_bronze_tipo'])} registros brutos do tipo na "
            f"Bronze colapsaram para só {int(linha_max['registros_silver'])} linhas na Silver. "
            f"Isso é dedup real por chave natural: esse tipo usa um identificador de entidade "
            f"física com cardinalidade baixa e fixa (poucas dezenas/centenas de valores "
            f"possíveis), então o MERGE INTO está colapsando leituras repetidas da mesma "
            f"entidade ao longo do tempo. Os demais tipos (dedup ≈ {linha_min['taxa_dedup_pct']:.0f}%) "
            f"usam identificador gerado por execução (sempre único), então não deduplicam — "
            f"comportamento esperado do modelo de dados, não um defeito."
        )
    print(
        f"\nTamanho físico e nº de arquivos ({desc['n_arquivos'].min()}-{desc['n_arquivos'].max()} "
        f"arquivos, {desc['tamanho_mb'].min():.2f}-{desc['tamanho_mb'].max():.2f} MB por tabela) e "
        f"nº de snapshots ({desc['n_snapshots'].min()}-{desc['n_snapshots'].max()} por tabela) "
        "confirmam que cada execução da Silver gera um snapshot Iceberg novo — o versionamento "
        "que sustenta o time travel está ativo (aqui observado em poucas execuções; a "
        "acumulação de snapshots ao longo de mais runs é o que a dag_iceberg_maintenance "
        "posteriormente compacta)."
    )
else:
    print("Sem dados de contagens ainda.")

Comentário sobre a tabela descritiva
Leitura da coluna 'taxa_dedup_pct': o denominador agora é a fatia da Bronze de cada tipo (total / n_tipos), então a taxa reflete deduplicação real por tipo, não o artefato de misturar 8 tipos numa Bronze só. Taxa ~0% = o tipo usa identificador gerado por execução (sempre único), então o MERGE INTO da Silver não tem o que colapsar; taxa alta = deduplicação real por chave natural.

'sensor' se destaca com 99.9% de dedup — 100000 registros brutos do tipo na Bronze colapsaram para só 105 linhas na Silver. Isso é dedup real por chave natural: esse tipo usa um identificador de entidade física com cardinalidade baixa e fixa (poucas dezenas/centenas de valores possíveis), então o MERGE INTO está colapsando leituras repetidas da mesma entidade ao longo do tempo. Os demais tipos (dedup ≈ 0%) usam identificador gerado por execução (sempre único), então não deduplicam — comportamento esperado do modelo de dados, não um defeito.

Tamanho físico e nº de arquivos 

In [ ]:
# 6. Sintese final — combina as leituras de E1, E2 e da tabela descritiva.
# Gerado a partir dos mesmos numeros calculados acima (nao e texto fixo).
print("## 6. Síntese e discussão geral\n")

n_rep = None
if len(e1):
    n_rep = int(e1_agg["count"].max())
    bronze_ct = e1_agg[e1_agg["dag"] == "dag_ingestao_bronze"].sort_values("volume_total")
    silver_ct = e1_agg[e1_agg["dag"] == "dag_silver_transform"].sort_values("volume_total")
    cresc_bronze = bronze_ct["mean"].iloc[-1] / bronze_ct["mean"].iloc[0]
    cresc_silver = silver_ct["mean"].iloc[-1] / silver_ct["mean"].iloc[0]
    print(
        f"**E1 (escalabilidade):** em 2 ordens de grandeza de volume (8×10³→8×10⁵ "
        f"registros), a arquitetura não quebra nem degrada abruptamente — o estágio mais "
        f"sensível ao volume (Bronze, leitura/escrita de arquivos) cresceu "
        f"{cresc_bronze:.1f}x, enquanto Silver/Gold ficaram essencialmente planos "
        f"({cresc_silver:.1f}x), dominados por overhead fixo de orquestração. Como a "
        f"duração cresce muito menos que o volume em todos os estágios, nesses volumes a "
        f"pipeline permanece no regime dominado por overhead: E1 é evidência de que a "
        f"arquitetura é robusta e previsível nesse regime, não de escalabilidade linear do "
        f"processamento (que exigiria volumes maiores — ver V4, Seção 7). O comportamento "
        f"contrário (crescimento explosivo ou não-linear) teria sido um sinal de alerta "
        f"que este experimento não encontrou."
    )
if len(e2) and "speedup" in e2_agg.columns:
    n_rep = int(e2_agg["count"].max())
    n_max = int(e2_agg["cores"].max())
    linha = e2_agg.loc[e2_agg["cores"] == n_max]
    speedup_max = linha["speedup"].iloc[0]
    ic_max = linha["speedup_ic95"].iloc[0]
    print(
        f"\n**E2 (paralelismo):** o speedup com mais núcleos foi modesto "
        f"({speedup_max:.2f}x ± {ic_max:.2f} com {n_max} núcleos) porque o gargalo da "
        f"Silver é orquestração (8 spark-submit sequenciais, cada um pagando startup fixo "
        f"de sessão), não processamento paralelo de dados. Reportar esse número honesto — "
        f"com seu IC 95%, em vez de omiti-lo ou escolher só a configuração mais "
        f"favorável — é consistente com a prática de HPC: eficiência baixa em cargas "
        f"pequenas dominadas por overhead de coordenação é esperado, não um defeito a "
        f"esconder."
    )
txt_reps = f"{n_rep} repetições por ponto" if n_rep else "repetições por ponto"
print(
    f"\n**Limitações:** hardware único (notebook de posto de comando, não cluster — "
    f"por isso o relatório reporta grandezas relativas), sem E2b (paralelismo "
    f"inter-task, ver Seção 7) e {txt_reps}. A duração é cronometrada em milissegundos "
    f"e a incerteza da média é reportada como IC 95% (t de Student) ao lado do "
    f"desvio-padrão; ainda assim, num ambiente tão determinístico (hardware único, "
    f"reset total entre reps) a dispersão observada é pequena, então o IC reflete mais "
    f"a estabilidade do ambiente do que variabilidade intrínseca da carga."
)


## 7. Limitações e próximos passos (opcional)

- **E2b (inter-task):** remover o encadeamento sequencial das 8 tasks da
  `dag_silver_transform` e subir `max_active_tasks`, medindo paralelismo no
  eixo de orquestração — pelo achado da Seção 4/6 (gargalo é overhead de sessão
  por task, não processamento), esse é o experimento com maior potencial de
  mostrar speedup real, já que paraleliza exatamente o que hoje é sequencial.
- **V4 (10⁶ registros):** como V1-V3 rodaram sem falhas e em tempo razoável,
  adicionar um quarto ponto de volume reforçaria a curva de escalabilidade da
  Figura E1 com mais um ponto na faixa alta.